# Práctica: Versicolor vs Virginica y Paramétricos vs No Paramétricos
Objetivos:
- Clasificar Versicolor vs Virginica usando solo sepal length y sepal width.
- Estandarizar variables con StandardScaler.
- Comparar métodos paramétricos (LDA, QDA) con no paramétricos (KDE y k-NN).
- Ver cómo cambian las fronteras al variar:
  - KDE: bandwidth en [0.2, 0.5, 0.8] y kernel en ['tophat', 'gaussian']
  - k-NN: k en [1, 3, 5, 7, 9]
- Evaluar accuracy y matriz de confusión en test.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KernelDensity, KNeighborsClassifier

from sklearn.metrics import accuracy_score, confusion_matrix

## 1) Carga, filtrado y preprocesamiento

In [ ]:
iris = load_iris()
X_all = iris.data
y_all = iris.target
names = iris.target_names

# Versicolor (1) y Virginica (2)
mask = (y_all == 1) | (y_all == 2)
X = X_all[mask][:, [0, 1]]
y_raw = y_all[mask]

# Reetiquetar: 0=Versicolor, 1=Virginica
y = np.where(y_raw == 1, 0, 1)

print("Clases usadas: 0 =", names[1], "| 1 =", names[2])
print("Observaciones:", X.shape[0])
print("X shape:", X.shape)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# estandarizar
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

## 2) Modelos paramétricos: LDA y QDA

In [ ]:
lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis()

lda.fit(X_train_s, y_train)
qda.fit(X_train_s, y_train)

def eval_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    acc = accuracy_score(y_te, pred)
    cm = confusion_matrix(y_te, pred, labels=[0,1])
    return {"model": name, "accuracy": acc, "confusion_matrix": cm}

res_lda = eval_model("LDA", lda, X_test_s, y_test)
res_qda = eval_model("QDA", qda, X_test_s, y_test)

print("LDA accuracy:", res_lda["accuracy"])
print(res_lda["confusion_matrix"])

print("\nQDA accuracy:", res_qda["accuracy"])
print(res_qda["confusion_matrix"])

## 3) Grid 2D para fronteras (en espacio estandarizado)

In [ ]:
def make_grid(X_s, step=0.02, pad=0.5):
    x_min, x_max = X_s[:,0].min() - pad, X_s[:,0].max() + pad
    y_min, y_max = X_s[:,1].min() - pad, X_s[:,1].max() + pad
    xx, yy = np.meshgrid(np.arange(x_min, x_max, step),
                         np.arange(y_min, y_max, step))
    grid = np.c_[xx.ravel(), yy.ravel()]
    return xx, yy, grid

xx, yy, grid = make_grid(np.vstack([X_train_s, X_test_s]))

In [ ]:
def plot_boundary(ax, title, predict_func, X_tr, y_tr):
    Z = predict_func(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3)
    ax.scatter(X_tr[:,0], X_tr[:,1], c=y_tr, s=18)
    ax.set_title(title)
    ax.set_xlabel("sepal length (scaled)")
    ax.set_ylabel("sepal width (scaled)")

## 4) KDE (Kernel Density Estimation) por clase
- Estimamos p(x|C0) y p(x|C1) con KernelDensity para cada clase.
- Probamos bandwidth = [0.2, 0.5, 0.8]
- Probamos kernels: 'tophat' y 'gaussian'
- Clasificación: g_i(x) = p(x|Ci) * P(Ci) y elegimos la clase con mayor g

In [ ]:
bandwidths = [0.2, 0.5, 0.8]
kernels = ["tophat", "gaussian"]

# separar train por clase
X_train_c0 = X_train_s[y_train == 0]
X_train_c1 = X_train_s[y_train == 1]

P_c0 = len(X_train_c0) / len(X_train_s)
P_c1 = len(X_train_c1) / len(X_train_s)

def kde_predict_factory(bw, kernel):
    kde0 = KernelDensity(bandwidth=bw, kernel=kernel).fit(X_train_c0)
    kde1 = KernelDensity(bandwidth=bw, kernel=kernel).fit(X_train_c1)

    def predict(X_points):
        log0 = kde0.score_samples(X_points)
        log1 = kde1.score_samples(X_points)
        p0 = np.exp(log0) * P_c0
        p1 = np.exp(log1) * P_c1
        return (p1 > p0).astype(int)  # 1 si clase 1 gana

    return predict

## 5) k-NN
- Entrenamos KNeighborsClassifier con k en los siguientes valores [1, 3, 5, 7, 9]
- Vemos cómo cambia la frontera al aumentar k

In [ ]:
ks = [1, 3, 5, 7, 9]
knn_models = {}

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    knn_models[k] = knn

## 6) Evaluación (accuracy + matriz de confusión) para todos los métodos

In [ ]:
all_results = []

# LDA/QDA
all_results.append(res_lda)
all_results.append(res_qda)

# KDE
for ker in kernels:
    for bw in bandwidths:
        name = f"KDE({ker}, bw={bw})"
        pred_func = kde_predict_factory(bw, ker)
        y_pred = pred_func(X_test_s)
        acc = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred, labels=[0,1])
        all_results.append({"model": name, "accuracy": acc, "confusion_matrix": cm})

# k-NN
for k, model in knn_models.items():
    name = f"kNN(k={k})"
    y_pred = model.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=[0,1])
    all_results.append({"model": name, "accuracy": acc, "confusion_matrix": cm})

df_acc = pd.DataFrame([{"model": r["model"], "accuracy": r["accuracy"]} for r in all_results])
df_acc = df_acc.sort_values("accuracy", ascending=False).reset_index(drop=True)

print("Tabla de accuracies en test:")
display(df_acc)

## 7) Fronteras de decisión (subplots)

In [ ]:
# LDA vs QDA
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_boundary(axes[0], "LDA", lda.predict, X_train_s, y_train)
plot_boundary(axes[1], "QDA", qda.predict, X_train_s, y_train)
plt.tight_layout()
plt.show()

In [ ]:
# KDE gaussian con distintos bw
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, bw in enumerate(bandwidths):
    pred_func = kde_predict_factory(bw, "gaussian")
    plot_boundary(axes[i], f"KDE gaussian bw={bw}", pred_func, X_train_s, y_train)
plt.tight_layout()
plt.show()

In [ ]:
# KDE tophat con distintos bw
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, bw in enumerate(bandwidths):
    pred_func = kde_predict_factory(bw, "tophat")
    plot_boundary(axes[i], f"KDE tophat bw={bw}", pred_func, X_train_s, y_train)
plt.tight_layout()
plt.show()

In [ ]:
# k-NN con distintos k
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, k in enumerate(ks):
    knn = knn_models[k]
    plot_boundary(axes[i], f"kNN k={k}", knn.predict, X_train_s, y_train)
plt.tight_layout()
plt.show()

## 8) Comentarios

Comparación numérica (test):
- El mejor método en este split es KDE(tophat, bw=0.8) con accuracy = 0.7333.
- LDA obtiene 0.7000 y QDA 0.6667.
- En k-NN el mejor fue k=7 con 0.6667 (k=1 y k=3: 0.6333; k=9: 0.5667).

Efecto de bandwidth en KDE:
- Con kernel tophat, bw pequeño (0.2 y 0.5) da peor rendimiento (0.4333 y 0.4667) y fronteras muy “recortadas”.
- Con bw grande (0.8) la frontera se suaviza y mejora (0.7333).
- Con kernel gaussian, al aumentar bw la frontera también se suaviza y el rendimiento sube hasta 0.7000 (bw=0.8).

Efecto de k en k-NN:
- k pequeño (1–3) genera fronteras más irregulares y aquí queda en 0.6333.
- k intermedio (7) es el mejor (0.6667).
- k grande (9) suaviza demasiado y empeora (0.5667).

Paramétricos vs no paramétricos:
- LDA y QDA imponen una forma fija de frontera. Aquí LDA generaliza mejor que QDA.
- KDE y k-NN dependen más del parámetro (bw o k): si el parámetro es demasiado pequeño hay más varianza generando uns frontera irregular y si es demasiado grande hay más sesgo generando una frontera demasiado suave.